# 🏥 VietMedKG — LightRAG Entities & Relationships → Qdrant

Notebook này nạp **Entities** và **Relationships** từ `preprocessed_data.csv` vào Qdrant Cloud,
giúp LightRAG chuyển từ chế độ `naive` (chỉ vector search trên chunks) sang chế độ `local`
(tìm kiếm ngữ nghĩa trên cả entities + relationships + chunks).

```
lightrag_vdb_chunks        ← đã có sẵn (7,552 points)
lightrag_vdb_entities      ← notebook này nạp vào
lightrag_vdb_relationships ← notebook này nạp vào
```

**ID format khớp 100% với LightRAG internals** (không cần sửa LightRAG source code).

## ⚙️ Checklist trước khi Run All:
- [ ] **Add Data** → Dataset `preprocessed_data.csv` đã được thêm
- [ ] **Accelerator** → GPU T4 x1 đã bật
- [ ] **Kaggle Secrets** → `QDRANT_URL` và `QDRANT_API_KEY` đã thêm

Ước tính: **~10-20 phút** (GPU T4, ~50,000 records)

In [ ]:
# ── Cell 1: Cài đặt ─────────────────────────────────────────────────────
import subprocess, sys

for pkg in ["qdrant-client", "sentence-transformers"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("✅ Done")

In [ ]:
# ── Cell 2: Cấu hình ─────────────────────────────────────────────────────
import os, glob

# --- Qdrant credentials ---
QDRANT_URL     = "https://dacb4228-6c2b-438f-98c7-3378b52d5b05.eu-west-2-0.aws.cloud.qdrant.io"
QDRANT_API_KEY = "PASTE_YOUR_API_KEY_HERE"  # ← thay bằng API key thật!

# Thử load từ Kaggle Secrets (override nếu có)
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    QDRANT_URL     = _s.get_secret("QDRANT_URL")
    QDRANT_API_KEY = _s.get_secret("QDRANT_API_KEY")
    print("✅ Credentials từ Kaggle Secrets")
except Exception as e:
    print(f"ℹ️  Kaggle Secrets không khả dụng ({e.__class__.__name__}), dùng giá trị hardcoded")

# --- Tự động tìm file CSV ---
csv_candidates = sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True))
CSV_PATH = next(
    (p for p in csv_candidates if "preprocessed_data" in p),
    csv_candidates[0] if csv_candidates else None,
)
if CSV_PATH is None:
    raise FileNotFoundError("Không tìm thấy file CSV! Hãy vào Add Data → thêm dataset preprocessed_data.csv")

# --- Các hằng số ---
ENTITY_COLLECTION      = "lightrag_vdb_entities"
REL_COLLECTION         = "lightrag_vdb_relationships"
EMBEDDING_DIM          = 1024
WORKSPACE_ID           = "_"    # DEFAULT_WORKSPACE của LightRAG
BATCH_SIZE             = 64     # GPU T4: giảm vì relationships text dài hơn

print(f"CSV          : {CSV_PATH}")
print(f"Entities col : {ENTITY_COLLECTION}")
print(f"Relations col: {REL_COLLECTION}")
print(f"Qdrant URL   : {QDRANT_URL[:55]}...")

In [ ]:
# ── Cell 3: Helper — ID functions khớp với LightRAG internals ───────────
import hashlib, uuid

def compute_mdhash_id(content: str, prefix: str = "") -> str:
    """Khớp với lightrag.utils.compute_mdhash_id."""
    return prefix + hashlib.md5(content.encode("utf-8")).hexdigest()

def make_qdrant_point_id(internal_id: str, workspace: str = "_") -> str:
    """SHA256(workspace + internal_id)[:16] → UUID hex. Khớp với qdrant_impl.compute_mdhash_id_for_qdrant."""
    hashed = hashlib.sha256((workspace + internal_id).encode("utf-8")).digest()
    return uuid.UUID(bytes=hashed[:16], version=4).hex

# Smoke test
_test_entity = "bệnh tiểu đường"
_internal_id = compute_mdhash_id(_test_entity, prefix="ent-")
_qdrant_id   = make_qdrant_point_id(_internal_id, WORKSPACE_ID)
print(f"Entity       : {_test_entity}")
print(f"Internal ID  : {_internal_id}")
print(f"Qdrant pt ID : {_qdrant_id}")
print("✅ ID functions OK")

In [ ]:
# ── Cell 4: Đọc CSV và xây dựng Entities + Relationships ────────────────
import pandas as pd
from collections import defaultdict

def _safe(val) -> str:
    if val is None or (isinstance(val, float) and val != val):
        return ""
    return str(val).strip()

df = pd.read_csv(CSV_PATH, encoding="utf-8")
print(f"📂 CSV: {len(df)} dòng — cột: {list(df.columns[:6])}")

# ── Xây dựng Entities ───────────────────────────────────────────────────
# entity_id (internal mdhash_id) → {content, entity_name, source_id}
entities: dict = {}

# Entity loại Bệnh (Disease)
# Entity loại Triệu chứng block (Symptom block — 1 entity per disease)
# Entity loại Điều trị block (Treatment block)
# Entity loại Lời khuyên block (Advice block)
# Entity loại Thuốc (Drug — deduplicated)
# Entity loại Khoa (Department — deduplicated)

drug_entities: dict = {}   # drug_name → description (deduplicated)
dept_entities: dict = {}   # dept_name → description (deduplicated)

for _, row in df.iterrows():
    name = _safe(row.get("disease_name"))
    if not name:
        continue

    desc     = _safe(row.get("disease_description"))
    category = _safe(row.get("disease_category"))
    cause    = _safe(row.get("disease_cause"))
    symptom  = _safe(row.get("disease_symptom"))
    cure     = _safe(row.get("cure_method"))
    dept     = _safe(row.get("cure_department"))
    drug_common  = _safe(row.get("drug_common"))
    drug_detail  = _safe(row.get("drug_detail"))
    nutrition_do = _safe(row.get("nutrition_do_eat"))
    prevention   = _safe(row.get("disease_prevention"))

    # -- 1. Disease entity --
    disease_desc = f"Mô tả: {desc}." if desc else ""
    if category:
        disease_desc += f" Danh mục: {category}."
    if cause:
        disease_desc += f" Nguyên nhân: {cause}."
    disease_entity_name = name
    disease_content = disease_entity_name + "\n" + disease_desc
    disease_eid = compute_mdhash_id(disease_entity_name, prefix="ent-")
    entities[disease_eid] = {
        "content": disease_content,
        "entity_name": disease_entity_name,
        "source_id": compute_mdhash_id(disease_entity_name),  # dùng disease_name làm chunk tham chiếu
    }

    # -- 2. Symptom block entity (per disease) --
    if symptom:
        sym_entity_name = f"{name}_symptoms"
        sym_desc = f"Triệu chứng của bệnh {name}: {symptom}."
        sym_eid = compute_mdhash_id(sym_entity_name, prefix="ent-")
        entities[sym_eid] = {
            "content": sym_entity_name + "\n" + sym_desc,
            "entity_name": sym_entity_name,
            "source_id": disease_eid,
        }

    # -- 3. Treatment block entity (per disease) --
    if cure:
        trt_entity_name = f"{name}_treatment"
        trt_desc = f"Phương pháp điều trị bệnh {name}: {cure}."
        trt_eid = compute_mdhash_id(trt_entity_name, prefix="ent-")
        entities[trt_eid] = {
            "content": trt_entity_name + "\n" + trt_desc,
            "entity_name": trt_entity_name,
            "source_id": disease_eid,
        }

    # -- 4. Advice block entity (per disease) --
    advice_parts = []
    if nutrition_do:
        advice_parts.append(f"Nên ăn: {nutrition_do}")
    if prevention:
        advice_parts.append(f"Phòng tránh: {prevention}")
    if advice_parts:
        adv_entity_name = f"{name}_advice"
        adv_desc = f"Lời khuyên cho bệnh {name}. " + ". ".join(advice_parts) + "."
        adv_eid = compute_mdhash_id(adv_entity_name, prefix="ent-")
        entities[adv_eid] = {
            "content": adv_entity_name + "\n" + adv_desc,
            "entity_name": adv_entity_name,
            "source_id": disease_eid,
        }

    # -- 5. Drug entities (deduplicated) --
    for drug_raw in [drug_common]:
        if drug_raw:
            for drug_name in [d.strip() for d in drug_raw.split(",") if d.strip()]:
                if drug_name not in drug_entities:
                    drug_entities[drug_name] = f"Thuốc: {drug_name}." + (f" Chi tiết: {drug_detail}." if drug_detail else "")

    # -- 6. Department entities (deduplicated) --
    if dept:
        for d_name in [d.strip() for d in dept.split(",") if d.strip()]:
            if d_name not in dept_entities:
                dept_entities[d_name] = f"Khoa điều trị: {d_name}."

# Add drug + dept entities
for drug_name, drug_desc in drug_entities.items():
    eid = compute_mdhash_id(drug_name, prefix="ent-")
    entities[eid] = {
        "content": drug_name + "\n" + drug_desc,
        "entity_name": drug_name,
        "source_id": compute_mdhash_id(drug_name),
    }

for dept_name, dept_desc in dept_entities.items():
    eid = compute_mdhash_id(dept_name, prefix="ent-")
    entities[eid] = {
        "content": dept_name + "\n" + dept_desc,
        "entity_name": dept_name,
        "source_id": compute_mdhash_id(dept_name),
    }

print(f"\n✅ Tổng entities: {len(entities):,}")
print("  - Disease entities:", sum(1 for v in entities.values() if '_symptoms' not in v['entity_name'] and '_treatment' not in v['entity_name'] and '_advice' not in v['entity_name']))
print("  - Symptom blocks:", sum(1 for v in entities.values() if '_symptoms' in v['entity_name']))
print("  - Treatment blocks:", sum(1 for v in entities.values() if '_treatment' in v['entity_name']))
print("  - Advice blocks:", sum(1 for v in entities.values() if '_advice' in v['entity_name']))

# --- Sample entity ---
first_key = next(iter(entities))
print(f"\n--- Sample Entity ---")
print(f"  ID      : {first_key}")
print(f"  name    : {entities[first_key]['entity_name']}")
print(f"  content : {entities[first_key]['content'][:120]}")

In [ ]:
# ── Cell 5: Xây dựng Relationships ──────────────────────────────────────
# rel_id (internal mdhash_id) → {src_id, tgt_id, content, keywords, description, weight}
relationships: dict = {}

def add_rel(src: str, tgt: str, keywords: str, description: str, weight: float = 1.0):
    """Thêm relationship. Khớp với LightRAG: key = mdhash(src+tgt, prefix='rel-')."""
    rel_id = compute_mdhash_id(src + tgt, prefix="rel-")
    content = f"{keywords}\t{src}\n{tgt}\n{description}"
    relationships[rel_id] = {
        "src_id": src,
        "tgt_id": tgt,
        "content": content,
        "keywords": keywords,
        "description": description,
        "weight": weight,
    }

for _, row in df.iterrows():
    name = _safe(row.get("disease_name"))
    if not name:
        continue

    symptom    = _safe(row.get("disease_symptom"))
    cure       = _safe(row.get("cure_method"))
    drug_common = _safe(row.get("drug_common"))
    associated = _safe(row.get("associated_disease"))
    nutrition_do = _safe(row.get("nutrition_do_eat"))
    prevention   = _safe(row.get("disease_prevention"))

    # 1. HAS_SYMPTOM: Disease → Symptom block
    if symptom:
        add_rel(
            src=name,
            tgt=f"{name}_symptoms",
            keywords="triệu chứng, symptom, biểu hiện",
            description=f"Bệnh {name} có triệu chứng: {symptom}.",
        )

    # 2. HAS_TREATMENT: Disease → Treatment block
    if cure:
        add_rel(
            src=name,
            tgt=f"{name}_treatment",
            keywords="điều trị, treatment, phác đồ",
            description=f"Bệnh {name} điều trị bằng: {cure}.",
        )

    # 3. IS_PRESCRIBED: Disease → Drug (many per disease)
    if drug_common:
        for drug_name in [d.strip() for d in drug_common.split(",") if d.strip()][:5]:  # top 5 drugs
            add_rel(
                src=name,
                tgt=drug_name,
                keywords="thuốc, drug, dược phẩm, điều trị",
                description=f"Bệnh {name} được điều trị bằng thuốc {drug_name}.",
            )

    # 4. HAS_ADVICE: Disease → Advice block
    if nutrition_do or prevention:
        add_rel(
            src=name,
            tgt=f"{name}_advice",
            keywords="lời khuyên, advice, dinh dưỡng, phòng tránh",
            description=f"Bệnh {name} có lời khuyên dinh dưỡng và phòng tránh.",
        )

    # 5. IS_LINKED_WITH: Disease → Associated Disease(s)
    if associated:
        for linked_disease in [d.strip() for d in associated.split(",") if d.strip()][:3]:  # top 3
            add_rel(
                src=name,
                tgt=linked_disease,
                keywords="bệnh liên quan, comorbidity, biến chứng",
                description=f"Bệnh {name} có liên quan hoặc đồng mắc với bệnh {linked_disease}.",
            )

print(f"✅ Tổng relationships: {len(relationships):,}")

# Sample
first_rel_key = next(iter(relationships))
r = relationships[first_rel_key]
print(f"\n--- Sample Relationship ---")
print(f"  ID       : {first_rel_key}")
print(f"  src_id   : {r['src_id']}")
print(f"  tgt_id   : {r['tgt_id']}")
print(f"  content  : {r['content'][:120]}")

In [ ]:
# ── Cell 6: Load BAAI/bge-m3 ─────────────────────────────────────────────
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Device: {device}")

print("⏳ Downloading BAAI/bge-m3 (~2.5GB, lần đầu mất 2-3 phút)...")
model = SentenceTransformer("BAAI/bge-m3", device=device)
print(f"✅ Loaded | dim = {model.get_sentence_embedding_dimension()}")

test_emb = model.encode(["Bệnh tiểu đường triệu chứng"], normalize_embeddings=True)
print(f"✅ Smoke test shape: {test_emb.shape}")

In [ ]:
# ── Cell 7: Kết nối Qdrant + setup collections ───────────────────────────
from qdrant_client import QdrantClient, models

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
print("✅ Qdrant connected")

def ensure_collection(col_name: str):
    """Tạo collection nếu chưa có, với đúng config như lightrag_vdb_chunks."""
    exists = client.collection_exists(col_name)
    print(f"Collection '{col_name}' exists: {exists}")
    if not exists:
        client.create_collection(
            collection_name=col_name,
            vectors_config=models.VectorParams(size=EMBEDDING_DIM, distance=models.Distance.COSINE),
            hnsw_config=models.HnswConfigDiff(payload_m=16, m=0),
        )
        print(f"  ✅ Created '{col_name}'")
    # Tạo workspace_id index (LightRAG cần để filter)
    try:
        client.create_payload_index(
            collection_name=col_name,
            field_name="workspace_id",
            field_schema=models.KeywordIndexParams(
                type=models.KeywordIndexType.KEYWORD, is_tenant=True
            ),
        )
        print(f"  ✅ workspace_id index ready")
    except Exception:
        print(f"  ℹ️  workspace_id index already exists")

ensure_collection(ENTITY_COLLECTION)
ensure_collection(REL_COLLECTION)

# Số records hiện có
ws_filter = models.Filter(must=[models.FieldCondition(key="workspace_id", match=models.MatchValue(value=WORKSPACE_ID))])
existing_entities = client.count(collection_name=ENTITY_COLLECTION, count_filter=ws_filter, exact=True).count
existing_rels     = client.count(collection_name=REL_COLLECTION,    count_filter=ws_filter, exact=True).count
print(f"\n📊 Existing entities     : {existing_entities:,} / {len(entities):,}")
print(f"📊 Existing relationships: {existing_rels:,} / {len(relationships):,}")

In [ ]:
# ── Cell 8: Helper — Upsert function ─────────────────────────────────────
import numpy as np, time
from tqdm.auto import tqdm

TIMESTAMP = int(time.time())

def upsert_to_qdrant(
    col_name: str,
    data: dict,
    meta_fields: set,
    batch_size: int = BATCH_SIZE,
):
    """
    Embed và upsert data vào Qdrant.

    data: {internal_id: {content, ...meta_fields}} — format giống lightrag upsert input
    Point payload format khớp với QdrantVectorDBStorage.upsert():
        {id, workspace_id, created_at, content, ...meta_fields}
    """
    items = list(data.items())
    total = len(items)
    success = 0
    errors  = 0
    total_batches = (total + batch_size - 1) // batch_size

    print(f"🚀 Upserting {total:,} records to '{col_name}' in {total_batches} batches")

    for i in tqdm(range(0, total, batch_size), total=total_batches, desc=col_name):
        batch = items[i : i + batch_size]
        internal_ids = [k for k, _ in batch]
        values       = [v for _, v in batch]
        texts        = [v["content"] for v in values]

        try:
            # Embed
            embs = model.encode(texts, normalize_embeddings=True, show_progress_bar=False, batch_size=32)

            # Build Qdrant points — payload format khớp QdrantVectorDBStorage
            points = []
            for j, (internal_id, v) in enumerate(zip(internal_ids, values)):
                payload = {
                    "id": internal_id,
                    "workspace_id": WORKSPACE_ID,
                    "created_at": TIMESTAMP,
                    "content": v["content"],
                }
                # Thêm các meta_fields
                for field in meta_fields:
                    if field in v:
                        payload[field] = v[field]

                qdrant_point_id = make_qdrant_point_id(internal_id, WORKSPACE_ID)
                points.append(
                    models.PointStruct(
                        id=qdrant_point_id,
                        vector=embs[j].tolist(),
                        payload=payload,
                    )
                )

            client.upsert(collection_name=col_name, points=points, wait=True)
            success += len(batch)

        except Exception as e:
            errors += len(batch)
            print(f"\n❌ Batch {i//batch_size+1} failed: {e}")

    print(f"\n{'='*50}")
    print(f"✅ Success : {success:,}")
    print(f"❌ Failed  : {errors:,}")
    print(f"{'='*50}")
    return success, errors

In [ ]:
# ── Cell 9: Upsert Entities ───────────────────────────────────────────────
# meta_fields khớp với lightrag.py dòng 716:
# meta_fields={"entity_name", "source_id", "content", "file_path"}
entity_meta_fields = {"entity_name", "source_id"}

success_e, errors_e = upsert_to_qdrant(
    col_name=ENTITY_COLLECTION,
    data=entities,
    meta_fields=entity_meta_fields,
    batch_size=BATCH_SIZE,
)

# Verify
final_entity_count = client.count(
    collection_name=ENTITY_COLLECTION,
    count_filter=ws_filter,
    exact=True,
).count
print(f"\n📊 Entities in Qdrant: {final_entity_count:,} / {len(entities):,}")
if final_entity_count >= len(entities):
    print("✅ ENTITIES COMPLETE!")
else:
    print(f"⚠️  Còn thiếu {len(entities) - final_entity_count:,} entities — chạy lại Cell này")

In [ ]:
# ── Cell 10: Upsert Relationships ─────────────────────────────────────────
# meta_fields khớp với lightrag.py dòng 722:
# meta_fields={"src_id", "tgt_id", "source_id", "content", "file_path"}
rel_meta_fields = {"src_id", "tgt_id", "keywords", "description", "weight"}

success_r, errors_r = upsert_to_qdrant(
    col_name=REL_COLLECTION,
    data=relationships,
    meta_fields=rel_meta_fields,
    batch_size=BATCH_SIZE,
)

# Verify
final_rel_count = client.count(
    collection_name=REL_COLLECTION,
    count_filter=ws_filter,
    exact=True,
).count
print(f"\n📊 Relationships in Qdrant: {final_rel_count:,} / {len(relationships):,}")
if final_rel_count >= len(relationships):
    print("✅ RELATIONSHIPS COMPLETE!")
else:
    print(f"⚠️  Còn thiếu {len(relationships) - final_rel_count:,} rels — chạy lại Cell này")

In [ ]:
# ── Cell 11: Verify toàn bộ + Semantic Search Test ────────────────────────
print("=" * 60)
print("📊 FINAL SUMMARY")
print("=" * 60)

for col_name, total in [(ENTITY_COLLECTION, len(entities)), (REL_COLLECTION, len(relationships))]:
    count = client.count(collection_name=col_name, count_filter=ws_filter, exact=True).count
    status = "✅" if count >= total else "⚠️ "
    print(f"  {status} {col_name}: {count:,} / {total:,}")

print()
# Chunks (giữ nguyên, chỉ báo cáo)
chunks_count = client.count(
    collection_name="lightrag_vdb_chunks",
    count_filter=ws_filter,
    exact=True,
).count
print(f"  ℹ️  lightrag_vdb_chunks   : {chunks_count:,} (không đổi)")

print()
print("--- Semantic Search Test (entities) ---")
for query in ["bệnh tiểu đường triệu chứng", "thuốc điều trị cao huyết áp", "viêm phổi điều trị"]:
    q_emb = model.encode([query], normalize_embeddings=True)[0]
    hits = client.query_points(
        collection_name=ENTITY_COLLECTION,
        query=q_emb.tolist(),
        limit=3,
        with_payload=True,
        score_threshold=0.2,
        query_filter=ws_filter,
    ).points
    print(f"\nQuery: '{query}'")
    for h in hits:
        print(f"  [{h.score:.3f}] {h.payload.get('entity_name', 'N/A')}")

print()
print("--- Semantic Search Test (relationships) ---")
for query in ["triệu chứng tiểu đường", "thuốc điều trị viêm phổi"]:
    q_emb = model.encode([query], normalize_embeddings=True)[0]
    hits = client.query_points(
        collection_name=REL_COLLECTION,
        query=q_emb.tolist(),
        limit=3,
        with_payload=True,
        score_threshold=0.2,
        query_filter=ws_filter,
    ).points
    print(f"\nQuery: '{query}'")
    for h in hits:
        print(f"  [{h.score:.3f}] {h.payload.get('src_id','?')} → {h.payload.get('tgt_id','?')}")

print("\n🎉 INGESTION COMPLETE! Hãy cập nhật .env và pipeline.py theo hướng dẫn.")